# Producers

## What's covered

- The shape of a record — what a `ProducerRecord` actually carries
- Producer architecture — accumulator, batches, sender thread
- The default (sticky) partitioner and the `murmur2` key hash
- Custom partitioners — when to write one and why you usually shouldn't
- `acks` — `0`, `1`, `all` — the durability-vs-latency dial
- `min.insync.replicas` — the broker-side guarantee that makes `acks=all` meaningful
- Retries — `retries`, `retry.backoff.ms`, `delivery.timeout.ms`
- `max.in.flight.requests.per.connection` and its quiet effect on ordering
- The idempotent producer — what it fixes and what it does not
- Transactional producer — atomic multi-topic writes (preview; full exactly-once story finishes in notebook 03)
- Batching — `batch.size` and `linger.ms` — throughput-vs-latency dial
- Compression — `gzip`, `snappy`, `lz4`, `zstd`
- `buffer.memory` and back-pressure
- Delivery callbacks — how you find out a record actually landed
- Common gotchas

## The shape of a record

Every record produced to Kafka is a small envelope with seven fields:

| Field | Required | Purpose |
|---|---|---|
| `topic` | yes | Where to send it |
| `value` | yes (may be null) | The payload — opaque bytes to the broker |
| `key` | no | Decides partition (via hash) and groups same-key records together |
| `partition` | no | Override the partitioner and pin to a specific partition |
| `timestamp` | no | Defaults to producer wall-clock (`CreateTime`); the broker can be configured to stamp with `LogAppendTime` instead |
| `headers` | no | Free-form key/value byte pairs — tracing IDs, schema IDs, tenant IDs |
| _(internal)_ offset | — | Assigned by the broker when the record lands; returned to the producer in the ack |

The broker treats `key` and `value` as opaque bytes — serialization is entirely a client concern (notebook 05). For now we use plain strings; `confluent-kafka` UTF-8-encodes them automatically.

## Producer architecture — what `produce()` actually does

`producer.produce(...)` does not send a network request. It enqueues the record in an in-memory buffer (the **record accumulator**) and returns immediately. A background **sender thread** drains the accumulator, groups records into per-partition **batches**, and sends each batch as a single produce request.

```text
  application thread          ┌──────────────────────────┐    sender thread
  ───────────────────         │   record accumulator     │    ────────────────
  produce(...)  ─────────────►│ ┌──────┐ ┌──────┐ ┌────┐ │───► group by partition
  produce(...)  ─────────────►│ │ p0   │ │ p1   │ │ p2 │ │     into batches
  produce(...)  ─────────────►│ │batch │ │batch │ │... │ │           │
                              │ └──────┘ └──────┘ └────┘ │           ▼
                              └──────────────────────────┘     leader broker(s)
```

Three knobs you will tune over and over:

- **`batch.size`** (default 16 KB) — the per-partition batch ceiling. Once a partition's batch reaches this size, the sender ships it immediately.
- **`linger.ms`** (default `0`) — how long the sender waits for more records before shipping a not-yet-full batch. Raising this trades a few milliseconds of latency for far more throughput, because batches are bigger and per-request overhead amortizes.
- **`buffer.memory`** (default 32 MB) — total memory the accumulator can hold across all partitions. When full, `produce()` blocks (or fails, depending on `max.block.ms`) until the sender drains it. This is your built-in back-pressure mechanism.

The takeaway: `produce()` is asynchronous. The record is acknowledged only when the sender's request to the broker returns successfully — which is why you need delivery callbacks.

## Setup

Same broker as notebook 01. We'll reuse `foundations-demo` and create a couple of new demo topics along the way.

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"
DEMO_TOPIC = "foundations-demo"   # created in notebook 01

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

# Idempotent helper — create-if-missing for any topic we need.
def ensure_topic(name, partitions=3, rf=1):
    futures = admin.create_topics([NewTopic(name, num_partitions=partitions, replication_factor=rf)])
    for n, f in futures.items():
        try:
            f.result()
            print(f"Created {n}")
        except Exception as e:
            if "TopicExistsError" in str(type(e)) or "already exists" in str(e):
                print(f"{n} already exists")
            else:
                raise

ensure_topic(DEMO_TOPIC)

## Delivery callbacks — how you know a record actually landed

Because `produce()` is asynchronous, the only way to learn a record's fate is a delivery callback. The callback receives an `err` and a `msg`:

- `err is None` → the broker acknowledged the write per the `acks` setting. `msg.partition()` and `msg.offset()` tell you where it landed.
- `err is not None` → the write failed after all retries were exhausted (or the record was rejected — payload too big, topic missing with auto-create off, ...).

Callbacks fire on the producer's internal poll thread. You trigger draining of pending callbacks by calling `producer.poll(0)` periodically or `producer.flush()` at the end.

**The most common mistake new producers make:** writing `producer.produce(...)` in a loop and exiting the script without `flush()`. The script terminates before the sender thread has shipped anything — records silently vanish from the buffer. Always `flush()`.

In [ ]:
def on_delivery(err, msg):
    if err is not None:
        print(f"FAILED: {err}")
        return
    print(f"  delivered  key={msg.key().decode():<10} -> partition={msg.partition()} offset={msg.offset()}")

producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",                # wait for every in-sync replica (covered below)
    "enable.idempotence": True,   # safe retries (covered below)
})

for key, value in [("CUST0001", "deposit  120.00"),
                   ("CUST0002", "withdraw  40.00"),
                   ("CUST0003", "transfer 200.00")]:
    producer.produce(DEMO_TOPIC, key=key, value=value, on_delivery=on_delivery)
    producer.poll(0)   # serve any callbacks ready to fire

producer.flush()   # block until everything is delivered (or fails)
print("all callbacks have fired")

## The default partitioner — sticky + key hash

When you produce a record, the partitioner decides which partition it lands on. The default `confluent-kafka` partitioner is **consistent random + sticky**, mirroring the modern Java client:

- **With a key:** `partition = murmur2(key) % num_partitions`. Same key → same partition, always (until `num_partitions` changes). The hash is `murmur2`, the same one the Java client uses — so a Python producer and a Java producer with the same key choose the same partition. That cross-language consistency is what lets you mix clients on the same topic.
- **Without a key:** the **sticky partitioner** picks one partition and keeps producing to it until its current batch is full or `linger.ms` expires, then picks another. Earlier clients round-robined per-record, which fragmented batches across all partitions and hurt throughput; sticky concentrates records into bigger batches.

Watch the partition assignment below: same-key records always land together, while no-key records cluster on one partition at a time.

In [ ]:
p = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

results = []
def collect(err, msg):
    if err is None:
        k = msg.key().decode() if msg.key() else "<none>"
        results.append((k, msg.partition()))

# Three records with the same key -> same partition.
for _ in range(3):
    p.produce(DEMO_TOPIC, key="CUST0001", value="x", on_delivery=collect)

# Three records with different keys -> hash spreads them.
for k in ["CUST0002", "CUST0003", "CUST0004"]:
    p.produce(DEMO_TOPIC, key=k, value="x", on_delivery=collect)

# Three records with no key -> sticky partitioner sends them to one partition.
for _ in range(3):
    p.produce(DEMO_TOPIC, value="x", on_delivery=collect)

p.flush()

print(f"{'key':<10}  partition")
print("-" * 22)
for k, part in results:
    print(f"{k:<10}  {part}")

## Custom partitioners

`confluent-kafka` lets you swap the partitioner via the `partitioner` config — supported values include `consistent_random` (default, sticky + hash), `consistent` (hash only, no sticky), `murmur2`, `murmur2_random`, and `fnv1a`. Use these when you need byte-for-byte compatibility with a non-default Java partitioner on the same topic.

Writing a fully custom partitioner in Python is not supported by the C client — for that, drop down to the Java client. In practice, you almost never need a custom partitioner. Two legitimate reasons:

- **Geographic routing** — pin records from each region to a dedicated set of partitions so a regional consumer reads only its own slice.
- **Tenant isolation** — give large tenants their own partitions so one heavy tenant cannot back-pressure the others.

Both can usually be solved instead by **encoding the routing information into the key** and letting the default partitioner do its job. That's almost always the cleaner answer.

## `acks` — the durability dial

When a producer sends a batch to a partition leader, how long does the broker wait before acknowledging? That's `acks`, and it has three settings:

| `acks` | Leader's behavior | Durability | Latency |
|---|---|---|---|
| `0` | Don't wait; ack immediately on the network | Lowest — record can vanish on broker crash | Lowest |
| `1` | Wait for the leader to write to its local log | Medium — survives consumer failure but not leader failure before replication | Medium |
| `all` (aka `-1`) | Wait for every in-sync replica to acknowledge | Highest — survives loss of any single broker (with `min.insync.replicas >= 2`) | Highest |

Two things to remember:

1. **`acks=all` alone is not enough.** If only the leader is in-sync (because all followers fell behind), `acks=all` collapses to `acks=1`. Pair it with broker-level `min.insync.replicas=2` (or higher) so that a write to a single-replica ISR is *rejected* instead of silently accepted.
2. **`acks=all` is the new default.** Since Kafka 3.0, both `acks` and `enable.idempotence` default to safe values (`all` and `true`). The unsafe `acks=0` and `acks=1` are now deliberate downgrades, not the path of least resistance.

For anything resembling production, use `acks=all`. Use `acks=1` only for high-volume telemetry where losing a few records on a broker crash is acceptable. `acks=0` is for fire-hose use cases where the system as a whole tolerates loss (clickstream sampling, metrics).

## Retries, timeouts, and ordering

When a produce request fails for a *retriable* reason — leader election in progress, broker briefly unavailable, network blip — the producer retries automatically. The settings that govern this:

- **`retries`** — number of retries. Defaults to a very large number (`Integer.MAX_VALUE` on the Java side). You almost never reduce this; bound the total time via the timeout below instead.
- **`retry.backoff.ms`** (default `100`) — delay between retries.
- **`delivery.timeout.ms`** (default `120000` = 2 min) — the upper bound on the **total time from `produce()` call to delivery success or failure**, including all retries. This is the right knob to tune; it makes the retry behavior bounded regardless of `retries`.
- **`request.timeout.ms`** (default `30000`) — per-request timeout for one broker round-trip. Must be less than `delivery.timeout.ms`.

And one config that quietly interacts with ordering:

- **`max.in.flight.requests.per.connection`** (default `5`) — how many produce requests can be unacknowledged on a single connection at once. Higher is faster, but with a non-idempotent producer **and** retries enabled, a retried request can land *after* a later request that succeeded the first time, reordering records within a partition. Setting it to `1` prevents that reordering for non-idempotent producers — at a steep throughput cost. Idempotent producers (next section) get up to `5` safely, because the broker deduplicates and reorders by sequence number on the wire.

## Idempotent producer — what it actually fixes

Set one config:

```python
Producer({"bootstrap.servers": ..., "enable.idempotence": True})
```

What it fixes: **duplicates caused by producer retries.** Before idempotence, a network blip between leader and producer could cause this sequence — broker writes record, ack lost in flight, producer retries, broker writes the record again. Same record, two offsets.

How it works: the producer is assigned a **Producer ID (PID)** on first connection and stamps every record with `(PID, sequence_number)`. The broker tracks the last sequence per `(PID, partition)` and rejects duplicates — same `(PID, sequence)` arrives twice, the second write is silently deduplicated.

What it does **not** fix:

- Application-level duplicates — if your code calls `produce()` twice, that's two distinct sequence numbers, two records.
- Cross-session duplicates — a producer that crashes and restarts gets a new PID. The broker has no way to recognize the same logical writer. Use **transactions** (next section) with a stable `transactional.id` for that.
- Cross-topic atomicity — idempotence is per-partition. "Either both these records land or neither" needs transactions.

**Default since Kafka 3.0.** When you enable idempotence, the producer also auto-enforces `acks=all`, `retries > 0`, and `max.in.flight.requests.per.connection <= 5`. If you set any of those to incompatible values, the producer raises a config error rather than starting up broken.

## Transactions — atomic multi-topic writes

Idempotence covers a single record on a single partition across producer retries. **Transactions** extend that to *a group of records, possibly across multiple topics and partitions, written atomically.* Consumers configured with `isolation.level=read_committed` see either all of them or none.

Two settings opt you in:

- **`transactional.id`** — a stable, application-chosen identifier. Same logical producer process across restarts should use the same `transactional.id`. The broker uses it to **fence off** older instances — if a new producer with the same `transactional.id` registers, any in-flight transaction from a previous instance is aborted. This is what makes transactions survive crashes cleanly.
- **`enable.idempotence=True`** — implied. Transactions build on the idempotent producer.

The lifecycle for one transaction:

1. `init_transactions()` once at startup — registers the `transactional.id`, fences any predecessor.
2. `begin_transaction()`
3. `produce(...)` to one or more topics/partitions
4. `commit_transaction()` **or** `abort_transaction()`

Below: write to two topics atomically. We commit the first transaction and abort the second; only committed records are visible to a `read_committed` consumer.

In [ ]:
ensure_topic("tx-orders", partitions=1)
ensure_topic("tx-payments", partitions=1)

tx_producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "transactional.id": "tx-demo-producer",   # stable ID across restarts
    # enable.idempotence and acks=all are implied when transactional.id is set.
})
tx_producer.init_transactions()

# Transaction 1 — commit.
tx_producer.begin_transaction()
tx_producer.produce("tx-orders",   key="ORD-001", value="order created")
tx_producer.produce("tx-payments", key="ORD-001", value="payment authorized")
tx_producer.commit_transaction()
print("transaction 1 committed: ORD-001 visible on both topics")

# Transaction 2 — abort. Records go to disk but are marked aborted; a
# read_committed consumer skips them.
tx_producer.begin_transaction()
tx_producer.produce("tx-orders",   key="ORD-002", value="order created")
tx_producer.produce("tx-payments", key="ORD-002", value="payment FAILED")
tx_producer.abort_transaction()
print("transaction 2 aborted: ORD-002 invisible to read_committed consumers")

In [ ]:
# Read both topics with isolation.level=read_committed and confirm only ORD-001 is visible.
c = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "tx-demo-reader",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
    "isolation.level": "read_committed",   # skip aborted records
})
c.subscribe(["tx-orders", "tx-payments"])

print(f"{'topic':<12}  key       value")
print("-" * 40)
seen = 0
while seen < 2:
    msg = c.poll(2.0)
    if msg is None: break
    if msg.error(): continue
    print(f"{msg.topic():<12}  {msg.key().decode():<8}  {msg.value().decode()}")
    seen += 1
c.close()

## Batching — `batch.size` and `linger.ms`

Two settings shape the throughput-vs-latency tradeoff:

- **`batch.size`** (default 16 KB) — the per-partition batch ceiling. When a batch reaches this size, it ships *immediately*, regardless of `linger.ms`.
- **`linger.ms`** (default `0`) — how long to wait *for more records* before shipping a not-yet-full batch.

With defaults (`linger.ms=0`), a slow trickle of records ships each one as its own request — high per-record network overhead, lots of small batches. Raising `linger.ms` to even `5` or `20` lets a few more records accumulate per batch, dramatically increasing throughput in exchange for that small added latency.

Below: send 5,000 small records twice — once with default linger, once with `linger.ms=20` — and compare the wall-clock time.

In [ ]:
import time

ensure_topic("batching-demo")

def time_produce(linger_ms, n=5000):
    p = Producer({
        "bootstrap.servers": BOOTSTRAP,
        "acks": "all",
        "enable.idempotence": True,
        "linger.ms": linger_ms,
    })
    start = time.perf_counter()
    for i in range(n):
        p.produce("batching-demo", value=f"msg-{i:05d}")
        if i % 1000 == 0:
            p.poll(0)
    p.flush()
    return time.perf_counter() - start

for lm in (0, 20):
    elapsed = time_produce(lm)
    print(f"linger.ms={lm:3d}  ->  {elapsed:6.2f}s  ({5000/elapsed:7.0f} records/s)")

## Compression — pay CPU, save bandwidth

Setting `compression.type` on the producer compresses each **batch** (not each record) before it leaves the client. The broker stores the batch compressed and the consumer decompresses it — so the savings extend through storage and replication too.

Four codecs ship with Kafka:

| Codec | Ratio | CPU | Notes |
|---|---|---|---|
| `gzip` | Best ratio | Highest CPU | Legacy default; pick only when bandwidth is the binding constraint |
| `snappy` | Modest ratio | Low CPU | The historical sweet spot |
| `lz4` | Modest ratio | Very low CPU | Faster than snappy on most workloads |
| `zstd` | Near-gzip ratio | Moderate CPU | The modern default — best ratio-to-CPU tradeoff. Requires brokers and clients on Kafka 2.1+ |

Compression *needs* batching to do anything useful — a one-record "batch" compresses badly because there's no redundancy to exploit. Pair `compression.type=zstd` with `linger.ms=10` or higher and watch the bytes-on-the-wire drop while throughput goes up.

Set it once on the producer and you're done — the broker and consumer figure it out from the batch header.

## `buffer.memory` and back-pressure

The accumulator's total size is bounded by **`buffer.memory`** (default 32 MB on the Java client; `queue.buffering.max.kbytes` on the C client, default 1 GB). When the buffer fills — typically because the broker is slow and the sender thread can't drain fast enough — `produce()` does one of two things, controlled by `max.block.ms`:

- Block until space is available (the default, for `max.block.ms` milliseconds), then
- Raise `BufferError` / `KafkaError` if still no space.

This is your built-in back-pressure mechanism: when the broker can't keep up, the producer eventually pushes back on the application. The opposite is worse — silently dropping records or growing memory without bound.

Two scenarios where you'll feel this:

- **Slow brokers, fast producer:** raise `linger.ms` or `compression.type` to make each request more efficient, or scale brokers.
- **Burst traffic:** the buffer absorbs short spikes for free. If spikes regularly exceed buffer size, either raise the buffer or pre-shape the input (rate-limit upstream).

## Common gotchas

Things that catch every new Kafka user at least once:

- **Forgetting `flush()`.** Your script exits with records still in the accumulator. They never ship. Always call `flush()` before the producer goes out of scope.
- **Ignoring delivery callbacks.** `produce()` returns success even when the eventual broker write fails. If you don't supply `on_delivery` (or check the eventual error somehow), failures are invisible.
- **Long-running callbacks.** Callbacks run on the producer's internal thread. Heavy work inside one blocks the sender; offload to a queue.
- **Changing partition count on a keyed topic.** The hash-to-partition mapping changes; same-key records produced before and after the change land on different partitions, breaking per-key ordering.
- **Producing to a topic that doesn't exist with auto-create off.** The first `produce()` fails with `UnknownTopicOrPartitionError`. Create topics explicitly with `AdminClient`.
- **Setting `acks=0` for anything that matters.** You will lose records on the next broker hiccup. Don't.
- **Reusing a `transactional.id` across two concurrent producers.** The second `init_transactions()` fences the first — the first will start seeing `ProducerFencedError`. `transactional.id` must be unique per producer instance you want to keep alive.

## What's next

You can now produce records safely — durably, idempotently, transactionally if needed, with a clear picture of the latency/throughput dials. Notebook 03 picks up on the other side:

- Consumer groups and the rebalance protocol
- Offset commits — auto vs manual, sync vs async, at-most-once vs at-least-once
- `auto.offset.reset` and what "earliest" vs "latest" actually mean for a new group
- Assignment strategies — range, round-robin, sticky, cooperative-sticky
- `isolation.level=read_committed` — the consumer half of the exactly-once story you started here

The producer's `transactional.id` plus a consumer's `read_committed` plus careful offset commits is the full exactly-once recipe — and the consumer notebook completes it.